# Sesión 4: Modelado de datos en Power BI## Conceptos, relaciones, cardinalidad y DAXEste notebook contiene conceptos clave y ejemplos de modelado de datos en Power BI para construir modelos relacionales robustos.

---## SLIDE 5: ¿Qué es el modelado de datos?### DefiniciónEl modelado de datos es el proceso de organizar datos en estructuras lógicas que reflejen relaciones entre entidades del mundo real.**Objetivos principales:**1. Mejorar la escalabilidad y eficiencia del modelo2. Separar datos operacionales (hechos) de datos descriptivos (dimensiones)3. Habilitar filtros, segmentaciones y visualizaciones precisas4. Evitar redundancias y errores de cálculo### Estructura Plana vs Modelo Relacional**Estructura Plana:**- Alta redundancia (mismo nombre de comuna repetido en cada fila)- Rendimiento decrece con volumen de datos- Capacidad de filtrado limitada- Dificultad para mantener cuando crece**Modelo Relacional:**- Baja redundancia (datos normalizados)- Mejor escalabilidad y rendimiento- Alta capacidad de filtrado (por relaciones)- Fácil de expandir modularmente

---## SLIDE 9-10: Tablas Fact y Tablas de Dimensión### Tabla de Hechos (Fact Table)**Definición:**Contiene los registros cuantificables, transacciones o eventos que pueden ser agregados.**Características:**- Cada fila representa una ocurrencia observable en el tiempo (una transacción)- Contiene claves foráneas que referencian dimensiones- Contiene métricas numéricas: cantidad, monto, tiempo, etc.- Alta granularidad (muchas filas)**Ejemplo - Tabla Solicitudes:**| ID_Solicitud | ID_Comuna | ID_Tipo_Servicio | Fecha_Recepcion | Tiempo_Resolucion ||--------------|-----------|------------------|-----------------|-------------------|| 1            | 1         | 1                | 2024-01-15      | 24                || 2            | 3         | 2                | 2024-01-16      | 48                || 3            | 1         | 1                | 2024-01-17      | 12                |### Tabla de Dimensión (Dimension Table)**Definición:**Contiene los atributos descriptivos asociados a los hechos.**Características:**- Contiene atributos textuales y categóricos- Una clave primaria única- Número mucho menor de filas- Permite filtrar, agrupar y segmentar los datos**Ejemplo - Tabla Comunas:**| ID_Comuna | Nombre_Comuna | Zona_Geografica | Region ||-----------|---------------|-----------------|--------|| 1         | Santiago      | Centro          | RM     || 2         | Providencia   | Centro          | RM     || 3         | La Florida    | Sur             | RM     |**Ejemplo - Tabla Tipos_Servicio:**| ID_Tipo_Servicio | Nombre_Servicio      | Area_Responsable ||------------------|----------------------|------------------|| 1                | Reparación vial      | Infraestructura  || 2                | Alumbrado público    | Servicios        |### Relación entre Fact y Dimension```Solicitudes (Fact) → ID_Comuna → Comunas (Dimension)Solicitudes (Fact) → ID_Tipo_Servicio → Tipos_Servicio (Dimension)```Al aplicar un filtro en Comunas, Power BI automáticamente filtra las Solicitudes que corresponden a esa comuna.

---## SLIDE 14-16: Tipos de Modelo: Estrella vs Copo de Nieve### Modelo en Estrella (Star Schema)**Estructura:**- Una única tabla de hechos en el centro- Se conecta DIRECTAMENTE con múltiples tablas de dimensiones- Las dimensiones están desnormalizadas (toda la información en una tabla)**Diagrama conceptual:**```                    Fechas                      |                      |Canales ─── Solicitudes ─── Comunas                      |                      |             Tipos_Solicitud```**Ventajas:**- Fácil de entender por usuarios no técnicos- Alto rendimiento en consultas DAX- Ideal para dashboards operativos- Queries rápidas**Desventajas:**- Puede haber redundancia si dimensiones tienen jerarquías complejas- No es óptimo para reutilización extrema de dimensiones**Recomendación:** Usar siempre que sea posible (90% de los casos)

### Modelo en Copo de Nieve (Snowflake Schema)**Estructura:**- Las dimensiones están normalizadas- Se dividen en subdimensiones o jerarquías- Reduce redundancia pero aumenta complejidad**Diagrama conceptual:**```            Regiones              |            Provincias              |            Comunas ─── Solicitudes ─── Canales                            |                      Tipos_Solicitud```**Ejemplo - Jerarquía Geográfica:**- Comunas tiene ID_Provincia- Provincias tiene ID_Region- Regiones es una tabla separada**Ventajas:**- Minimiza duplicación de datos- Útil cuando dimensiones se reutilizan en múltiples contextos**Desventajas:**- Consultas más lentas por múltiples saltos- Mayor dificultad de comprensión- Requiere más mantenimiento**Cuándo usar:**- Cuando las dimensiones tienen jerarquías complejas- En entornos empresariales muy grandes- Cuando se requiere máxima reutilización### Comparación - Criterios de elección| Criterio | Estrella | Copo de Nieve ||----------|----------|---------------|| Facilidad de uso | Alta | Media/Baja || Rendimiento | Alto | Medio || Escalabilidad | Media | Alta || Complejidad de mantenimiento | Baja | Alta || Contexto recomendado | Dashboards operativos | Reportes analíticos complejos |

---## SLIDE 20-22: Llaves Primarias y Foráneas### Llave Primaria (Primary Key)**Definición:**Campo (o conjunto de campos) que identifica de forma única cada fila dentro de una tabla.**Reglas:**- NO puede contener valores nulos- NO puede haber duplicados- DEBE ser única para cada registro**Ejemplo:**```Tabla Comunas:ID_Comuna (Llave Primaria) | Nombre_Comuna1                          | Santiago2                          | Providencia3                          | Ñuñoa```**Buenas prácticas:**- Usar números enteros pequeños (más eficientes)- Evitar usar campos de texto largo- Nombrar claramente (ID_Comuna, ID_Tipo)### Llave Foránea (Foreign Key)**Definición:**Campo en una tabla que apunta a la llave primaria de otra tabla.**Regla:**- Puede tener duplicados (muchas solicitudes de la misma comuna)**Ejemplo:**```Tabla Solicitudes:ID_Solicitud | ID_Comuna (Llave Foránea) | Fecha1            | 1                         | 2024-01-152            | 1                         | 2024-01-16  ← Mismo ID_Comuna3            | 3                         | 2024-01-17```### Relación entre Llaves```Comunas[ID_Comuna] (Primaria) ──────► Solicitudes[ID_Comuna] (Foránea)        ↑    Una sola vez    (ID único)                                          ↓                                    Puede repetirse                                    (muchas solicitudes)```### Validación en Power BIAntes de crear una relación, verifica:1. **Unicidad de la llave primaria:** `COUNTROWS(DISTINCT(Comunas[ID_Comuna]))` debe igual COUNTROWS(Comunas)2. **Que no haya nulos:** Ni en la llave primaria ni en la foránea3. **Tipos de datos coincidan:** Ambas como Int64 o ambas como Text4. **Claves huérfanas:** Valores en foránea que no existen en primaria

---## SLIDE 23-25: Relaciones entre tablas y Cardinalidad### Tipos de Relaciones**1. Uno a Uno (1:1)**- Cada fila en una tabla coincide con una sola fila en otra- Menos común- Ejemplo: Funcionarios ↔ Cuentas_Correo```Funcionarios          Cuentas_CorreoID_Funcionario  ──┬─→ ID_FuncionarioJuanPerez       ├─→ juan@municipio.clMariaDias       ├─→ maria@municipio.cl```**Cuándo usar:**- Dividir información muy sensible en tablas separadas- Compartimentalizar datos con diferentes niveles de acceso**2. Uno a Muchos (1:N)** ← MÁS COMÚN- Una fila de la tabla primaria se asocia con muchas filas en la tabla secundaria- Este es el tipo preferido para modelos dimensionales```Comunas                    SolicitudesID_Comuna (1) ──────►──┬─→ ID_SolicitudSantiago      (1)      ├─→ ID_Comuna (muchas)                       ├─→ Solicitud 1                       ├─→ Solicitud 2                       └─→ Solicitud N```**Cuándo usar:**- Siempre que sea posible (modelo estrella típico)- Dimensión → Tabla de Hechos- Permite filtrado cruzado efectivo**3. Muchos a Muchos (N:N)**- Varias filas en una tabla se relacionan con múltiples filas en otra- Requiere tratamiento especial en Power BI```Productos                    Campañas_DescuentoID_Producto ──┬─────────────┬─→ ID_Campana              ├─→ Producto A │   Campaña 1              └─→ Producto B ├─→ Campaña 2                             └─→ Campaña 3(Un producto en múltiples campañas)(Una campaña aplica a múltiples productos)```**Cómo evitarla o resolverla:**- Crear tabla puente intermedia- Usar valores derivados si es necesario- Evitar si es posible (impacta rendimiento)### Buenas prácticas de Cardinalidad1. **Preferir 1:N siempre**2. **Verificar la unicidad antes de crear relación**3. **Documentar la relación en el modelo**4. **Usar dirección de filtro correcta**5. **Evitar relaciones inactivas sin justificación**### Errores comunes❌ Intentar N:N sin tabla puente❌ Crear 1:1 innecesariamente (impide análisis con múltiples registros)❌ Relacionar columnas con tipos distintos (texto vs número)❌ Ignorar la dirección del filtro cruzado

---## SLIDE 26-28: Buenas Prácticas para Modelado### 1. Estructura del Modelo**✓ Preferir modelo en estrella**- Una tabla fact central- Múltiples dimensiones relacionadas directamente**✓ Separar claramente hechos de dimensiones**- Hechos: transacciones, eventos medibles- Dimensiones: categorías, atributos descriptivos**✓ Evitar columnas derivadas innecesarias**- Solo incluir lo que es necesario para el análisis### 2. Relacionamiento entre Tablas**✓ Definir correctamente llaves primarias**- Deben ser únicas- Sin nulos- Preferiblemente números enteros**✓ Verificar unicidad de claves en dimensiones**```// En Power Query o DAXDuplicados = COUNTROWS(Comunas) - COUNTROWS(DISTINCT(Comunas[ID_Comuna]))```**✓ Mantener dirección de filtros coherente**- Siempre de Dimensión → Fact- Evitar bidireccionales sin necesidad### 3. Nomenclatura y Documentación**✓ Usar nombres significativos**- ID_Comuna, Fecha_Cierre, Solicitudes_Totales- NO usar: Col1, Col2, X, Y**✓ Documentar el modelo**- Añadir descripciones de campos en Power BI Desktop- Crear diagrama visual del modelo**✓ Usar carpetas para organizar**- Carpeta "Medidas"- Carpeta "Dimensiones"- Carpeta "Hechos"### 4. Rendimiento y Escalabilidad**✓ Eliminar columnas innecesarias**- Especialmente texto libre muy largo**✓ Usar enteros como claves relacionales**- Mucho más eficiente que texto**✓ Evitar relaciones N:N sin control**- Impactan rendimiento significativamente### 5. Visualización y Usabilidad**✓ Ocultar campos técnicos**- ID_Comuna, ID_Tipo no necesitan verse en reportes**✓ Crear jerarquías de tiempo**- Año → Trimestre → Mes → Día- Facilita navegación en visualizaciones**✓ Nombrar campos de forma amigable**- Para usuarios finales- Usar títulos en lugar de nombres técnicos

---## SLIDE 30-33: Columnas Calculadas vs Medidas en DAX### Columnas Calculadas**Definición:**Campos nuevos que se agregan a una tabla, calculando un valor fila a fila.**Características:**- Se crean en Power BI Desktop (DAX)- Se calculan una sola vez al cargar el modelo- Se almacenan en la tabla (consumen memoria)- Se pueden usar en filtros y relaciones**Ejemplo 1 - Clasificación simple:**```daxClasificación_Duracion = IF(    Solicitudes[Tiempo_Resolucion] > 48,    "Larga",    "Corta")```**Ejemplo 2 - Concatenación:**```daxCodigo_Solicitud = Solicitudes[ID_Comuna] & "-" & TEXT(Solicitudes[Fecha_Solicitud], "YYYYMM") & "-" &FORMAT(Solicitudes[ID_Solicitud], "0000")```**Resultado:** "1-202401-0001", "3-202401-0002"**Ejemplo 3 - Cálculo condicional anidado:**```daxPrioridad_Operativa = IF(    Solicitudes[Tiempo_Resolucion] < 24,    "Crítica",    IF(        Solicitudes[Tiempo_Resolucion] < 48,        "Alta",        IF(            Solicitudes[Tiempo_Resolucion] < 72,            "Normal",            "Baja"        )    ))```**Cuándo usar columnas calculadas:**- ✓ Clasificar o categorizar registros- ✓ Crear jerarquías de navegación- ✓ Derivaciones que se usarán en relaciones- ✗ NO para agregaciones (usar medidas)### Medidas**Definición:**Expresiones dinámicas que realizan cálculos agregados sobre los datos.**Características:**- Se crean en Power BI Desktop (DAX)- Se recalculan en tiempo real según contexto- NO ocupan memoria por fila- Son muy eficientes- Solo devuelven un único valor**Ejemplo 1 - Contar registros:**```daxTotal_Solicitudes = COUNTROWS(Solicitudes)```**Resultado:** 1.256 (número total de solicitudes)**Ejemplo 2 - Promedio:**```daxPromedio_Duracion = AVERAGE(Solicitudes[Tiempo_Resolucion])```**Resultado:** 42.5 (días promedio)**Ejemplo 3 - Suma con contexto:**```daxTotal_Solicitudes_Rapidas = CALCULATE(    COUNTROWS(Solicitudes),    Solicitudes[Tiempo_Resolucion] < 24)```Cuenta solo solicitudes resueltas en menos de 24 horas.**Ejemplo 4 - Porcentaje de cumplimiento:**```daxTasa_Cumplimiento = VAR Total = [Total_Solicitudes]VAR Rapidas = [Total_Solicitudes_Rapidas]RETURN    DIVIDE(Rapidas, Total, 0)```**Ejemplo 5 - Variación mes a mes:**```daxVariacion_MesMes = VAR Mes_Actual = [Total_Solicitudes]VAR Mes_Anterior =     CALCULATE(        [Total_Solicitudes],        PREVIOUSMONTH(Fechas[Fecha])    )RETURN    DIVIDE(        Mes_Actual - Mes_Anterior,        Mes_Anterior,        0    )```**Cuándo usar medidas:**- ✓ Realizar agregaciones (suma, promedio, conteo)- ✓ Análisis dinámicos en visualizaciones- ✓ Crear KPIs- ✓ Cálculos que dependen de contexto de filtro### Comparación Rápida| Aspecto | Columna Calculada | Medida ||---------|------------------|--------|| Cálculo | Fila a fila | Agregado || Tiempo de cálculo | Una vez al cargar | En tiempo real || Uso de memoria | Sí (una por fila) | No (solo resultado) || Rendimiento | Rápido | Muy rápido || Flexibilidad | Baja | Alta || Contexto de filtro | No | Sí (muy importante) |### Ejemplo de Uso Integrado```dax// COLUMNA CALCULADA: preparar datosClasificacion = IF([Tiempo_Resolucion] > 48, "Larga", "Corta")// MEDIDA 1: contar totalTotal = COUNTROWS(Solicitudes)// MEDIDA 2: contar solo las "Largas"Total_Largas = CALCULATE(    [Total],    Solicitudes[Clasificacion] = "Larga")// MEDIDA 3: porcentajePorcentaje_Largas = DIVIDE([Total_Largas], [Total], 0)```En un gráfico: **Porcentaje_Largas** se muestra y se actualiza automáticamente al filtrar por Comuna o Tipo de Servicio.

---## Resumen de Conceptos Clave### Estructura de un Modelo1. **Tablas Fact** → Transacciones, eventos, métricas2. **Tablas Dimension** → Categorías, atributos3. **Relaciones** → Llaves primarias/foráneas4. **Medidas** → Cálculos dinámicos5. **Columnas calculadas** → Clasificaciones, derivaciones### Tipos de Modelo- **Estrella (90%)** → Simple, rápido, recomendado- **Copo de nieve** → Normalizado, reutilizable, más complejo### Cardinalidad- **1:1** → Raro- **1:N** → Estándar (preferido)- **N:N** → Evitar o usar tabla puente### Buenas Prácticas✓ Usar números enteros como claves✓ Verificar unicidad de llaves primarias✓ Documentar relaciones✓ Nombrar campos significativamente✓ Usar medidas para agregaciones✓ Usar columnas calculadas para clasificaciones✓ Preferir modelo en estrella✓ Evitar relaciones bidireccionales innecesarias

---## Recursos adicionales**Para verificar modelo:**- Vista de Modelo en Power BI Desktop- Herramienta de análisis de dependencias- Uso de DAX Studio para validar medidas**Documentación oficial:**- Microsoft Learn: Data modeling in Power BI- DAX Function Reference- Power BI best practices**Próximos pasos:**- Crear medidas avanzadas con CALCULATE- Usar variables VAR en DAX- Crear jerarquías de tiempo- Implementar seguridad a nivel de fila (RLS)